# Exploring the generated `.parquet` data

This notebook is standalone — it reads the data you saved to Google Drive.
**Run the mount cell first**, then set `DATA` to the Drive folder that holds
`runs/*.parquet` and `manifest.csv`.

In [ ]:
# 0. Mount Google Drive (the generated data was saved here)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import glob, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

DATA = "/content/drive/MyDrive/sewer_blockage_data"   # <-- folder with runs/ and manifest.csv
files = sorted(glob.glob(os.path.join(DATA, "runs", "*.parquet")))
print("DATA =", DATA)
print(len(files), "run files")
print(*[os.path.basename(f) for f in files[:8]], sep="\n")
assert files, "No parquet found - check the DATA path (look in your Drive for the folder you saved)."

## 1. Peek at one run (schema + first rows)

In [ ]:
df = pd.read_parquet(files[0])
print("shape:", df.shape, "  scenario:", df["scenario"].iloc[0], "  target:", df["target_conduit"].iloc[0])
print("\ncolumns:")
print(list(df.columns))
df.head()

## 2. Manifest overview (one row per run, with sampled parameters)

In [ ]:
man = pd.read_csv(os.path.join(DATA, "manifest.csv"))
print("runs in manifest:", len(man))
cols = [c for c in ["run_id","scenario","target","final_sev","ramp_type",
                    "duration_h","n_normal","n_rainfall","n_blockage"] if c in man.columns]
man[cols]

## 3. Class balance across ALL runs (timestep level)

In [ ]:
labels = pd.concat([pd.read_parquet(f, columns=["label"]) for f in files], ignore_index=True)["label"]
counts = labels.value_counts()
print(counts)
print("\npercent:\n", (100*counts/counts.sum()).round(2))
counts.reindex(["normal","rainfall","blockage"]).plot(kind="bar", color=["#bbbbbb","#4a90d9","#d9534f"])
plt.title("Class balance (timesteps, all runs)"); plt.ylabel("timesteps"); plt.tight_layout(); plt.show()

## 4. Plot a run: depth + flow at the primary sensor, shaded by label

In [ ]:
LABEL_COLORS = {"normal": "#cccccc", "rainfall": "#4a90d9", "blockage": "#d9534f"}

def plot_run(path):
    df = pd.read_parquet(path)
    tgt = df["target_conduit"].iloc[0]
    dcol = next(c for c in df.columns if c.startswith("depth__node_"))
    fcol = f"flow__{tgt}"
    fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    ax[0].plot(df["t_min"], df[dcol], color="black", lw=1); ax[0].set_ylabel("depth (m)")
    ax[1].plot(df["t_min"], df[fcol], color="black", lw=1); ax[1].set_ylabel("flow (cms)")
    ax[1].set_xlabel("minute")
    lab = df["label"].values; t = df["t_min"].values
    start = 0
    for i in range(1, len(lab) + 1):
        if i == len(lab) or lab[i] != lab[start]:
            for a in ax:
                a.axvspan(t[start], t[i-1], color=LABEL_COLORS.get(lab[start], "white"), alpha=0.18)
            start = i
    ax[0].set_title(f"{df['run_id'].iloc[0]}  |  scenario {df['scenario'].iloc[0]}  |  target {tgt}")
    handles = [plt.Rectangle((0,0),1,1,color=c,alpha=0.4) for c in LABEL_COLORS.values()]
    ax[0].legend(handles, LABEL_COLORS.keys(), ncol=3, loc="upper right", fontsize=8)
    plt.tight_layout(); plt.show()

plot_run(files[0])

Plot a specific run by id (edit `RUN_ID`):

In [ ]:
RUN_ID = "s3_r000"
match = [f for f in files if os.path.basename(f).startswith(RUN_ID)]
plot_run(match[0]) if match else print("no run named", RUN_ID, "- available:",
                                       [os.path.basename(f) for f in files[:10]])

## 5. Sensor-realism: clean vs measured (noisy + dropout) channel

In [ ]:
df = pd.read_parquet(files[0])
meas = [c for c in df.columns if c.endswith("_meas")]
if meas:
    c = meas[0]; base = c[:-5]
    plt.figure(figsize=(12, 3))
    plt.plot(df["t_min"], df[base], label="clean", lw=1.2)
    plt.plot(df["t_min"], df[c], label="measured (noise/drift/dropout)", lw=0.8, alpha=0.8)
    plt.title(base); plt.xlabel("minute"); plt.legend(); plt.tight_layout(); plt.show()
    print("measured NaNs (dropout):", int(df[c].isna().sum()))
else:
    print("no *_meas columns in this run")

## 6. Build a combined training table (all runs stacked)

In [ ]:
full = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print("combined shape:", full.shape)
print("runs:", full["run_id"].nunique(), "| scenarios:", sorted(full["scenario"].unique()))
full[["run_id","scenario","t_min","label","label_id","gt_severity"]].head()